# 1 - Parse CIDER VDJ Output

**Purpose**: Filter and summarise CIDER immune receptor clonotypes into per-sample,
per-locus CDR3 metrics (unique count, total reads, Shannon entropy, clonality).

The filter → group → summarise → zero-fill logic is implemented once as
`build_cider_base()` in `helpers/shortcuts.r`, so it can be reused for
alternate CDR3-confidence filters without duplication — see
`1b_cider_by_confidence.ipynb`, which builds PASS-only and
low-confidence-only variants from the same raw file.

| | |
|---|---|
| **Inputs** | `CIDER_DIR/cider.vdj.csv` |
| **Outputs** | `CIDER_PROJ_DIR/cider_base.csv` |
| **Run before** | `3_meta.ipynb`, `4_base.ipynb` |

In [1]:
source(paste0(dirname(getwd()),'/helpers/map.r'))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))

assert_files_exist(
  "CIDER VDJ file" = paste0(CIDER_DIR, "cider.vdj.csv")
)

# 0 - CIDER values

In [ ]:
# Raw CIDER VDJ output, unfiltered — build_cider_base() (helpers/shortcuts.r)
# applies the `filter` selection itself.
cider_vdj_raw <- fread(paste0(CIDER_DIR, "cider.vdj.csv"))

- All sample Ids

In [3]:
# all_samples: samples that satisfy all three conditions:
#   1. Present in clinical_update.txt     — have clinical data
#   2. Present in PURPLE_DIR/purity.csv — WGS was run (collected by 0_collect.ipynb)
#   3. Present in raw CIDER output — CIDER was run (at least one sequence detected,
#                                    even if all failed QC filters)
# This ensures zero-fill only applies to true zero-infiltration samples where
# CIDER ran but found no passing sequences, not samples where CIDER never ran.
clinical_ids <- fread(paste0(READY_DIR, "clinical_update.txt")) |>
  select(sampleId) |>
  distinct()

# Use collected PURPLE purity file — confirms WGS was run (from 0_collect.ipynb)
wgs_ids <- fread(paste0(PURPLE_DIR, "purity.csv")) |>
  select(sampleId) |>
  distinct()

# Reuse cider_vdj_raw (already loaded above) rather than re-reading
# cider.vdj.csv from disk a second time.
cider_ids <- cider_vdj_raw |>
  select(sampleId) |>
  distinct()

all_samples <- clinical_ids |>
  inner_join(wgs_ids,   by = "sampleId") |>
  inner_join(cider_ids, by = "sampleId")

- Build per-sample, per-group CDR3 metrics (primary filter: PASS + low-confidence)

In [ ]:
# Primary pipeline: keep both high-confidence ("PASS") and low-confidence
# ("NO_HIGH_QUAL_SUPPORT") CDR3 calls, as in previous CIDER releases.
# build_cider_base() filters, groups by lymphocyte subset, summarises
# (unique/total CDR3 counts, Shannon entropy, clonality), and zero-fills
# samples where CIDER ran but nothing passed the filter.
cider_base <- build_cider_base(
  cider_vdj_raw,
  filter_values = c("PASS", "NO_HIGH_QUAL_SUPPORT"),
  all_samples   = all_samples
)

- Append ImmuneLENS

In [ ]:
# ImmuneLENS fractions — joined if the combined output file exists.
# The file is produced by collect_ImmuneLENS/collect_results.R after all
# samples have been processed. Pipeline runs without it; the column will
# be NA until ImmuneLENS collection is complete.
#
# The collection script (collect_ImmuneLENS/1_run_immunelens.R) also runs
# ImmuneLENS on the IGH (B-cell) locus for potential future use, but this
# project is T-cell-focused (helpers/config.r's GP_LEVELS), so IGH rows are
# deliberately left unmapped below (case_when() returns NA for them) and
# drop out harmlessly at the left_join in the next cell.
il_path <- paste0(CIDER_PROJ_DIR, "immunelens_update.csv")

if (file.exists(il_path)) {
  immunelens <- fread(il_path) %>%
    transmute(
      sampleId,
      gp = case_when(
        locus == "TCRA" ~ "T-cell (TRA/TRD)",
        locus == "TCRB" ~ "T-cell (TRB)",
        locus == "TCRG" ~ "T-cell (TRG)"),
      immuneLENS_fraction = fraction)

  immunelensReady <- rbind(
    immunelens,
    immunelens %>%
      filter(grepl("T-cell", gp)) %>%
      mutate(gp = "T-cell") %>%
      group_by(sampleId, gp) %>%
      summarise(immuneLENS_fraction = mean(immuneLENS_fraction), .groups = "drop"))
} else {
  message("immunelens_update.csv not found — immuneLENS_fraction will be NA")
  immunelensReady <- tibble(sampleId = character(), gp = character(),
                            immuneLENS_fraction = numeric())
}

In [10]:
cider_base <- cider_base %>% left_join(immunelensReady, by = c("sampleId", "gp"))

# 2 - Send them

In [ ]:
fwrite(cider_base, paste0(CIDER_PROJ_DIR, "cider_base.csv"))